# 📖 Notebook 1: Geospatial Search for Nearby Businesses

The most important feature of Yelp is **finding businesses near you**. When a user searches "pizza near me," the system needs to quickly find businesses within a geographic radius.

The problem? Traditional database indexes (B-trees) don't work well for 2D spatial data like latitude/longitude. This notebook explores **why** and shows you **what does work**.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why a naive lat/lon query is slow
- How PostGIS geospatial indexes solve this
- How Elasticsearch geo_distance queries work
- How to cache geospatial search results in Redis
- How to combine location + category filters

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/yelp
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `yelp_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [1]:
import psycopg2
import psycopg2.extras
import redis
import json
import time
from elasticsearch import Elasticsearch

# -- Connection settings --
DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "yelp_demo", "user": "demo", "password": "demo"
}
REDIS_CONFIG = {"host": "localhost", "port": 6379, "decode_responses": True}
ES_URL = "http://localhost:9200"

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

def get_es():
    return Elasticsearch(ES_URL)

# Test connections
try:
    conn = get_db(); conn.close()
    print("✅ PostgreSQL connected")
except Exception as e:
    print(f"❌ PostgreSQL: {e}\n   Run: docker-compose up -d")

try:
    r = get_redis(); r.ping()
    print("✅ Redis connected")
except Exception as e:
    print(f"❌ Redis: {e}\n   Run: docker-compose up -d")

try:
    es = get_es(); es.info()
    print("✅ Elasticsearch connected")
except Exception as e:
    print(f"❌ Elasticsearch: {e}\n   Run: docker-compose up -d")

✅ PostgreSQL connected
✅ Redis connected
✅ Elasticsearch connected


## 🤔 The Problem: Finding Nearby Businesses

Imagine a user in **Manhattan** (latitude 40.758, longitude -73.985) searching for restaurants within **2 km**.

The naive approach: check every business and calculate distance.

```sql
-- This is SLOW — full table scan on every query!
SELECT * FROM businesses
WHERE latitude > 40.74 AND latitude < 40.78
  AND longitude > -74.00 AND longitude < -73.97;
```

**Why is this bad?**
- B-tree indexes handle **one dimension** well, but latitude + longitude is **two dimensions**
- The database can use the index for latitude OR longitude, but not both efficiently
- At 10M businesses, this means scanning millions of rows

Let's see this in action.

In [2]:
# The NAIVE approach: bounding box with plain columns
# This works but doesn't scale well

conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# User is in Manhattan
user_lat, user_lon = 40.758, -73.985
# Rough bounding box for ~2 km (0.018 degrees latitude ≈ 2 km)
delta = 0.018

start = time.time()
cur.execute("""
    SELECT id, name, city, latitude, longitude, avg_rating, num_reviews
    FROM businesses
    WHERE latitude BETWEEN %s AND %s
      AND longitude BETWEEN %s AND %s
    ORDER BY avg_rating DESC
    LIMIT 10;
""", (user_lat - delta, user_lat + delta, user_lon - delta, user_lon + delta))
naive_results = cur.fetchall()
naive_time = (time.time() - start) * 1000

print(f"📍 Naive bounding box search: {naive_time:.2f} ms")
print(f"   Found {len(naive_results)} businesses near Manhattan\n")
for b in naive_results[:5]:
    print(f"   ⭐ {b['avg_rating']} | {b['name']} ({b['city']})")

# Show the query plan — notice it doesn't use a spatial index
cur.execute("""
    EXPLAIN ANALYZE
    SELECT id, name, city, latitude, longitude
    FROM businesses
    WHERE latitude BETWEEN %s AND %s
      AND longitude BETWEEN %s AND %s;
""", (user_lat - delta, user_lat + delta, user_lon - delta, user_lon + delta))
plan = cur.fetchall()
print("\n📊 Query Plan (naive):")
for row in plan:
    print(f"   {row['QUERY PLAN']}")

conn.close()

📍 Naive bounding box search: 24.37 ms
   Found 10 businesses near Manhattan

   ⭐ 3.50 | Mountain Creek Repairs (New York)
   ⭐ 3.33 | Mountain Plaza Restaurant (New York)
   ⭐ 3.30 | Bay Bridge Repairs (New York)
   ⭐ 3.25 | Urban Creek Repairs (New York)
   ⭐ 3.23 | Central Corner Repairs (New York)

📊 Query Plan (naive):
   Seq Scan on businesses  (cost=0.00..37.00 rows=3 width=50) (actual time=0.035..0.070 rows=17 loops=1)
     Filter: ((latitude >= '40.74'::double precision) AND (latitude <= '40.776'::double precision) AND (longitude >= '-74.003'::double precision) AND (longitude <= '-73.967'::double precision))
     Rows Removed by Filter: 483
   Planning Time: 0.051 ms
   Execution Time: 0.134 ms


## 🌍 Better: PostGIS Geospatial Index

Our `init.sql` created a **geography** column with a **GIST index** (a spatial index).

PostGIS uses an **R-tree** under the hood — a data structure designed for multi-dimensional data. It groups nearby points into bounding rectangles, making it extremely fast to find points within a radius.

```
B-tree (1D):  ─────────────────────────────────►
              Only good for one dimension at a time

R-tree (2D):  ┌──────┐  ┌────┐
              │ •  • │  │ •  │    Groups nearby points
              │   •  │  │  • │    into rectangles
              └──────┘  └────┘
```

The key function is `ST_DWithin(location, point, distance_in_meters)` — it finds all points within a given radius using the spatial index.

In [3]:
# The POSTGIS approach: use ST_DWithin with the geography column

conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

user_lat, user_lon = 40.758, -73.985  # Manhattan
radius_meters = 2000  # 2 km

start = time.time()
cur.execute("""
    SELECT
        id, name, city, avg_rating, num_reviews, price_range,
        ST_Distance(location, ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography) AS distance_m
    FROM businesses
    WHERE ST_DWithin(
        location,
        ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography,
        %s
    )
    ORDER BY distance_m ASC
    LIMIT 10;
""", (user_lon, user_lat, user_lon, user_lat, radius_meters))
postgis_results = cur.fetchall()
postgis_time = (time.time() - start) * 1000

print(f"📍 PostGIS ST_DWithin search: {postgis_time:.2f} ms")
print(f"   Found {len(postgis_results)} businesses within {radius_meters}m of Manhattan\n")

for b in postgis_results:
    dist_km = b['distance_m'] / 1000
    price = '$' * (b['price_range'] or 1)
    print(f"   📍 {dist_km:.1f} km | ⭐ {b['avg_rating']} ({b['num_reviews']} reviews) | {price} | {b['name']}")

# Show the query plan — now it uses the GIST spatial index!
cur.execute("""
    EXPLAIN ANALYZE
    SELECT id, name
    FROM businesses
    WHERE ST_DWithin(
        location,
        ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography,
        %s
    );
""", (user_lon, user_lat, radius_meters))
plan = cur.fetchall()
print("\n📊 Query Plan (PostGIS):")
for row in plan:
    print(f"   {row['QUERY PLAN']}")

print("\n💡 Notice: PostGIS uses an Index Scan (GIST) instead of a Sequential Scan!")

conn.close()

📍 PostGIS ST_DWithin search: 70.21 ms
   Found 10 businesses within 2000m of Manhattan

   📍 0.2 km | ⭐ 0.00 (0 reviews) | $$$ | Blue Bridge Repairs
   📍 0.3 km | ⭐ 0.00 (0 reviews) | $$$ | Blue Corner Repairs
   📍 0.5 km | ⭐ 3.13 (15 reviews) | $ | Urban Phoenix Restaurant
   📍 0.6 km | ⭐ 0.00 (0 reviews) | $$ | Valley Springs Restaurant
   📍 0.7 km | ⭐ 3.50 (14 reviews) | $$ | Mountain Park Repairs
   📍 0.9 km | ⭐ 3.58 (12 reviews) | $$$$ | Grand Garden Repairs
   📍 1.2 km | ⭐ 3.22 (9 reviews) | $$$$ | Royal Place Repairs
   📍 1.4 km | ⭐ 3.23 (13 reviews) | $$ | Central Corner Repairs
   📍 1.4 km | ⭐ 2.89 (9 reviews) | $$$ | Downtown Dragon Repairs
   📍 1.6 km | ⭐ 0.00 (0 reviews) | $$ | Mountain Creek Restaurant

📊 Query Plan (PostGIS):
   Index Scan using idx_businesses_location on businesses  (cost=0.27..20.78 rows=1 width=24) (actual time=0.039..0.056 rows=13 loops=1)
     Index Cond: (location && _st_expand('0101000020E6100000D7A3703D0A7F52C01B2FDD2406614440'::geography, '2000':

## 🔍 Elasticsearch Geo-Distance Search

For production systems at Yelp's scale (10M businesses), **Elasticsearch** is the go-to choice. It natively supports:
- `geo_distance` queries (find points within a radius)
- `geo_bounding_box` queries (find points in a rectangle)
- Full-text search on business names
- Combined filters in a single query

Let's index our businesses into Elasticsearch and compare.

In [4]:
# Step 1: Create an Elasticsearch index with a geo_point mapping

es = get_es()

INDEX_NAME = "businesses"

# Delete the index if it exists (for re-running this notebook)
if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)

# Create the index with proper mappings
es.indices.create(
    index=INDEX_NAME,
    body={
        "mappings": {
            "properties": {
                "name":        {"type": "text"},        # full-text searchable
                "description": {"type": "text"},
                "city":        {"type": "keyword"},     # exact match
                "category":    {"type": "keyword"},     # exact match
                "location":    {"type": "geo_point"},   # geospatial!
                "avg_rating":  {"type": "float"},
                "num_reviews": {"type": "integer"},
                "price_range": {"type": "integer"}
            }
        }
    }
)
print("✅ Created Elasticsearch index with geo_point mapping")

✅ Created Elasticsearch index with geo_point mapping


In [5]:
# Step 2: Load businesses from Postgres into Elasticsearch

conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("""
    SELECT b.id, b.name, b.description, b.city, b.latitude, b.longitude,
           b.avg_rating, b.num_reviews, b.price_range, c.name AS category
    FROM businesses b
    JOIN categories c ON b.category_id = c.id
""")
businesses = cur.fetchall()
conn.close()

# Bulk index into Elasticsearch
from elasticsearch.helpers import bulk

actions = []
for biz in businesses:
    actions.append({
        "_index": INDEX_NAME,
        "_id": biz["id"],
        "_source": {
            "name": biz["name"],
            "description": biz["description"],
            "city": biz["city"],
            "category": biz["category"],
            "location": {"lat": float(biz["latitude"]), "lon": float(biz["longitude"])},
            "avg_rating": float(biz["avg_rating"] or 0),
            "num_reviews": biz["num_reviews"] or 0,
            "price_range": biz["price_range"]
        }
    })

success, errors = bulk(es, actions)
es.indices.refresh(index=INDEX_NAME)
print(f"✅ Indexed {success} businesses into Elasticsearch")
if errors:
    print(f"⚠️  {len(errors)} errors")

✅ Indexed 500 businesses into Elasticsearch


In [6]:
# Step 3: Search for nearby businesses using Elasticsearch geo_distance

user_lat, user_lon = 40.758, -73.985  # Manhattan

start = time.time()
result = es.search(
    index=INDEX_NAME,
    body={
        "query": {
            "bool": {
                "filter": [
                    {
                        "geo_distance": {
                            "distance": "2km",
                            "location": {"lat": user_lat, "lon": user_lon}
                        }
                    }
                ]
            }
        },
        "sort": [
            {
                "_geo_distance": {
                    "location": {"lat": user_lat, "lon": user_lon},
                    "order": "asc",
                    "unit": "km"
                }
            }
        ],
        "size": 10
    }
)
es_time = (time.time() - start) * 1000

print(f"📍 Elasticsearch geo_distance search: {es_time:.2f} ms")
print(f"   Found {result['hits']['total']['value']} businesses within 2km\n")

for hit in result["hits"]["hits"]:
    src = hit["_source"]
    dist = hit["sort"][0]  # distance in km
    price = '$' * (src.get('price_range') or 1)
    print(f"   📍 {dist:.1f} km | ⭐ {src['avg_rating']:.1f} ({src['num_reviews']} reviews) | {price} | {src['name']}")

📍 Elasticsearch geo_distance search: 2.70 ms
   Found 18 businesses within 2km

   📍 0.4 km | ⭐ 0.0 (0 reviews) | $$$ | Blue Dragon Repairs
   📍 0.5 km | ⭐ 3.2 (13 reviews) | $$ | Central Corner Repairs
   📍 0.6 km | ⭐ 3.1 (17 reviews) | $ | Grand Point Restaurant
   📍 0.7 km | ⭐ 0.0 (0 reviews) | $$ | Golden Dragon Restaurant
   📍 0.7 km | ⭐ 2.5 (11 reviews) | $$ | Green Meadow Repairs
   📍 0.8 km | ⭐ 3.5 (8 reviews) | $$ | Mountain Creek Repairs
   📍 0.8 km | ⭐ 0.0 (0 reviews) | $$ | Urban Park Repairs
   📍 0.9 km | ⭐ 0.0 (0 reviews) | $$$$ | Silver Springs Repairs
   📍 1.0 km | ⭐ 0.0 (0 reviews) | $ | Grand Terrace Restaurant
   📍 1.0 km | ⭐ 3.0 (12 reviews) | $$ | Happy Terrace Restaurant


## 🔗 Combining Location + Category Filters

In the real Yelp, users search for **"coffee shops near me"** — combining location proximity with a category filter. Elasticsearch handles this with a `bool` query that stacks multiple conditions.

In [7]:
# Combined search: restaurants within 5 km of San Francisco, sorted by rating

sf_lat, sf_lon = 37.7749, -122.4194  # San Francisco

result = es.search(
    index=INDEX_NAME,
    body={
        "query": {
            "bool": {
                "must": [
                    {"term": {"category": "Restaurants"}}  # exact category match
                ],
                "filter": [
                    {
                        "geo_distance": {
                            "distance": "5km",
                            "location": {"lat": sf_lat, "lon": sf_lon}
                        }
                    }
                ]
            }
        },
        "sort": [
            {"avg_rating": {"order": "desc"}},
            {
                "_geo_distance": {
                    "location": {"lat": sf_lat, "lon": sf_lon},
                    "order": "asc",
                    "unit": "km"
                }
            }
        ],
        "size": 10
    }
)

print(f"🍽️  Restaurants within 5 km of San Francisco (sorted by rating):")
print(f"   Total matches: {result['hits']['total']['value']}\n")

for hit in result["hits"]["hits"]:
    src = hit["_source"]
    dist = hit["sort"][1]  # geo_distance is second sort field
    price = '$' * (src.get('price_range') or 1)
    print(f"   ⭐ {src['avg_rating']:.1f} | 📍 {dist:.1f} km | {price} | {src['name']}")

🍽️  Restaurants within 5 km of San Francisco (sorted by rating):
   Total matches: 0



## ⚡ Caching Geospatial Search Results in Redis

Search results for popular locations ("restaurants in Manhattan") are requested thousands of times per minute. We can cache these results in Redis to avoid hitting Elasticsearch repeatedly.

**Strategy**: Use a cache key based on the search parameters. Set a short TTL (e.g., 60 seconds) since business data changes slowly but search rankings may need to stay fresh.

In [8]:
r = get_redis()

def search_nearby_cached(lat, lon, radius_km, category=None, limit=10):
    """
    Search for nearby businesses with Redis caching.
    Cache key encodes the search parameters so identical searches hit the cache.
    """
    # Build a cache key from the search parameters
    # Round lat/lon to 3 decimal places (~100m precision) to improve cache hit rate
    cache_key = f"search:{round(lat,3)}:{round(lon,3)}:{radius_km}:{category or 'all'}:{limit}"

    # Check cache first
    cached = r.get(cache_key)
    if cached:
        return json.loads(cached), True  # (results, was_cached)

    # Cache miss — query Elasticsearch
    query_body = {
        "query": {"bool": {"filter": [{"geo_distance": {"distance": f"{radius_km}km", "location": {"lat": lat, "lon": lon}}}]}},
        "sort": [{"avg_rating": {"order": "desc"}}, {"_geo_distance": {"location": {"lat": lat, "lon": lon}, "order": "asc", "unit": "km"}}],
        "size": limit
    }
    if category:
        query_body["query"]["bool"]["must"] = [{"term": {"category": category}}]

    result = es.search(index=INDEX_NAME, body=query_body)

    # Format results
    results = []
    for hit in result["hits"]["hits"]:
        src = hit["_source"]
        results.append({
            "name": src["name"], "category": src["category"],
            "avg_rating": src["avg_rating"], "num_reviews": src["num_reviews"],
            "distance_km": round(hit["sort"][1], 2)
        })

    # Store in cache with 60-second TTL
    r.setex(cache_key, 60, json.dumps(results))

    return results, False

# First call: cache MISS — hits Elasticsearch
start = time.time()
results, cached = search_nearby_cached(40.758, -73.985, 5, category="Restaurants")
t1 = (time.time() - start) * 1000
print(f"🔍 First search: {t1:.2f} ms (cached: {cached})")

# Second call: cache HIT — reads from Redis
start = time.time()
results, cached = search_nearby_cached(40.758, -73.985, 5, category="Restaurants")
t2 = (time.time() - start) * 1000
print(f"⚡ Second search: {t2:.2f} ms (cached: {cached})")
print(f"\n🚀 Speedup: {t1/t2:.1f}×\n")

for biz in results[:5]:
    print(f"   ⭐ {biz['avg_rating']:.1f} | 📍 {biz['distance_km']} km | {biz['name']}")

🔍 First search: 4.35 ms (cached: False)
⚡ Second search: 0.31 ms (cached: True)

🚀 Speedup: 14.0×

   ⭐ 3.8 | 📍 3.31 km | Royal Place Restaurant
   ⭐ 3.6 | 📍 2.33 km | Central Meadow Restaurant
   ⭐ 3.5 | 📍 3.39 km | Mountain Harbor Restaurant
   ⭐ 3.4 | 📍 2.44 km | Royal Park Restaurant
   ⭐ 3.4 | 📍 4.17 km | Royal Corner Restaurant


## 📊 Comparison: All Three Approaches

Let's measure them side by side.

In [9]:
# Benchmark: 50 identical searches per approach

def bench(label, fn, n=50):
    times = []
    for _ in range(n):
        start = time.time()
        fn()
        times.append((time.time() - start) * 1000)
    avg = sum(times) / len(times)
    print(f"  {label:<35} avg={avg:>7.2f} ms  min={min(times):>7.2f} ms  max={max(times):>7.2f} ms")
    return avg

print("⏱️  Benchmark: find restaurants within 2 km of Manhattan (50 runs each)\n")

lat, lon = 40.758, -73.985

def naive_search():
    c = get_db(); cr = c.cursor()
    cr.execute("SELECT id, name FROM businesses WHERE latitude BETWEEN %s AND %s AND longitude BETWEEN %s AND %s LIMIT 10",
               (lat-0.018, lat+0.018, lon-0.018, lon+0.018))
    cr.fetchall(); c.close()

def postgis_search():
    c = get_db(); cr = c.cursor()
    cr.execute("SELECT id, name FROM businesses WHERE ST_DWithin(location, ST_SetSRID(ST_MakePoint(%s,%s),4326)::geography, 2000) LIMIT 10",
               (lon, lat))
    cr.fetchall(); c.close()

def es_search():
    es.search(index=INDEX_NAME, body={"query":{"bool":{"filter":[{"geo_distance":{"distance":"2km","location":{"lat":lat,"lon":lon}}}]}},"size":10})

# Warm the Redis cache first
r.delete(f"search:{round(lat,3)}:{round(lon,3)}:2:all:10")
search_nearby_cached(lat, lon, 2)  # populates cache

def redis_cached_search():
    search_nearby_cached(lat, lon, 2)

t_naive = bench("Naive bounding box (Postgres)", naive_search)
t_postgis = bench("PostGIS ST_DWithin (Postgres)", postgis_search)
t_es = bench("Elasticsearch geo_distance", es_search)
t_redis = bench("Redis cached search", redis_cached_search)

print(f"\n💡 Key insight: caching makes search effectively instant for repeated queries.")
print(f"   In a real system you'd use Elasticsearch for fresh queries and Redis for hot results.")

⏱️  Benchmark: find restaurants within 2 km of Manhattan (50 runs each)



  Naive bounding box (Postgres)       avg=  57.14 ms  min=  45.82 ms  max=  90.08 ms


  PostGIS ST_DWithin (Postgres)       avg= 105.51 ms  min=  96.15 ms  max= 112.77 ms
  Elasticsearch geo_distance          avg=   1.62 ms  min=   1.33 ms  max=   3.95 ms
  Redis cached search                 avg=   0.29 ms  min=   0.19 ms  max=   0.43 ms

💡 Key insight: caching makes search effectively instant for repeated queries.
   In a real system you'd use Elasticsearch for fresh queries and Redis for hot results.


## 🧹 Cleanup

In [10]:
# Clean up Elasticsearch index and Redis cache keys
r = get_redis()
keys = r.keys("search:*")
if keys:
    r.delete(*keys)
    print(f"🧹 Cleaned {len(keys)} Redis cache keys")

# We keep the ES index for the next notebooks
print("🧹 Cleanup complete (ES index kept for next notebooks)")

🧹 Cleaned 2 Redis cache keys
🧹 Cleanup complete (ES index kept for next notebooks)


## 📚 Summary

### Key Takeaways

1. **Naive lat/lon queries are slow** — B-tree indexes can't efficiently handle 2D range queries
2. **PostGIS GIST indexes** use R-trees to make spatial queries fast — great for moderate scale
3. **Elasticsearch geo_distance** is purpose-built for search at scale — supports complex multi-filter queries
4. **Redis caching** makes repeated searches near-instant — use short TTLs to stay fresh
5. **Filter by distance first** — it's usually the most restrictive filter, shrinking the search space fastest

### System Design Interview Tips

- Start with PostGIS if the interviewer discourages Elasticsearch
- Mention **geohashing** and **quadtrees** as alternative spatial indexing strategies
- Always discuss the **second-pass filter**: use the Haversine formula to calculate exact distance after the index narrows the search space

### Next Up

In **Notebook 2**, we'll tackle **Review & Rating Aggregation** — how to efficiently calculate and update average ratings as reviews come in.